# Object Detection Chapter 11 Workshop 1

## Load module

In [20]:
import numpy as np
import cv2

cv2.__version__

'4.11.0'

## Load YOLO

In [ ]:
weight_path = "../models/yolov3/yolov3.weights"
cfg_path = "../models/yolov3/yolov3.cfg"
coco_names_path = "../models/yolov3/coco.names"

# Load the YOLO model
net = cv2.dnn.readNet(weight_path, cfg_path) # read files

# Create a list of class names from the coco.names file
classes = []
with open(coco_names_path, "r") as f:
    classes = [line.strip() for line in f.readlines()]

layer_names = net.getLayerNames()


print(f"Number of layer {len(layer_names)}")

output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

Number of layer 254


### check class YOLO

In [22]:
print(f"Number of classes: {len(classes)}")
# Print the output layers
print(f"Output layers: {output_layers}")
# Print the class names
print(f"Class names: {classes}")
# Print the number of layers in the network
print(f"Number of layers: {len(layer_names)}")
# Print the names of the layers
print(f"Layer names: {layer_names}")
# Print the type of the network
print(f"Network type: {type(net)}")

Number of classes: 80
Output layers: ['yolo_82', 'yolo_94', 'yolo_106']
Class names: ['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
Number of layers: 254
Layer names: ('conv_0', 'bn

In [23]:
def get_input_size_from_cfg(cfg_path):
    width, height = None, None
    with open(cfg_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('width='):
                width = int(line.split('=')[1])
            elif line.startswith('height='):
                height = int(line.split('=')[1])
            if width and height:
                break
    return (width, height)

# Usage
input_size = get_input_size_from_cfg(cfg_path)
print("Expected input size:", input_size)

Expected input size: (608, 608)


## Define color for each label

In [24]:
colors = np.random.uniform(0, 255, size=(len(classes), 3))

## Load image

In [25]:
# img_path = r"D:\Project\Learning\TensorFlow_Files\datasets\test_set\cat9023.jpg"
img_path = "../datasets/test_set/dog9040.jpg"  # Adjusted path for relative access
img = cv2.imread(img_path)
height, width, channels = img.shape
if img is None:
    print(f"Error: Could not read image from {img_path}")
else:
    if height > input_size[0] or width > input_size[1]:
        print(f"Warning: Image size {height}x{width} exceeds expected input size {input_size[0]}x{input_size[1]}. Resizing may be needed.")
        # Resize the image 
        # img = cv2.resize(img, (416, 416))
        img = cv2.resize(img, None,fx=0.8, fy=0.8)
print(f"Load image complete, image size: {img.shape}")
    


Load image complete, image size: (332, 500, 3)


In [26]:
img_size = input_size
# Create a blob from the image
blob = cv2.dnn.blobFromImage(img, 1/255.0, img_size, (0,0,0), True, crop=False)
net.setInput(blob)                  # Set the image input
# Forward pass through the network                          
output = net.forward(output_layers) # Get the output from the output layers

print(f"Output shape: {len(output)}")
print(type(output))

Output shape: 3
<class 'tuple'>


## Show output

In [ ]:
class_ids = []
confidences = []
boxes = []

# Loop through each output layer
for out in output:
    for detection in out:
        scores = detection[5:]          # Get the scores for each class
        class_id = np.argmax(scores)    # Get the index of the class with the highest score
        confidence = scores[class_id]   # Get the confidence score for that class
        
        if confidence > 0.5:            # Threshold to filter weak detections
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)
            
            # Calculate the coordinates of the bounding box
            x = int(center_x - w / 2)
            y = int(center_y - h / 2)
            
            boxes.append([x, y, w, h])  # Append the box coordinates
            confidences.append(float(confidence))  # Append the confidence score
            class_ids.append(class_id)  # Append the class ID

# Apply Non-Maximum Suppression to remove overlapping boxes
indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)    ## index of the label in image
print(f"class_ids: {class_ids}")
print(f"indices: {indices}")
print(f"confidences : {confidences}")
print(f"boxes: {boxes}")


class_ids: [0, 0, 0, 0, 0, 16, 16, 16, 16, 16, 16, 16, 16, 16]
indices: [8 4]
confidences : [0.6022272109985352, 0.712883710861206, 0.5570515990257263, 0.7235254049301147, 0.8892955183982849, 0.9809942841529846, 0.7795338034629822, 0.5965837240219116, 0.996354341506958, 0.9371206164360046, 0.515022337436676, 0.9931150078773499, 0.8887966871261597, 0.6515312194824219]
boxes: [[1, -1, 186, 92], [-1, 2, 187, 120], [-2, 4, 147, 154], [-5, 5, 184, 143], [-5, 1, 194, 158], [174, 47, 215, 210], [178, 46, 229, 212], [126, 49, 263, 235], [173, 62, 218, 211], [180, 60, 223, 216], [123, 55, 268, 251], [169, 65, 225, 229], [183, 65, 219, 231], [170, 78, 224, 231]]


In [30]:
text = [classes[class_ids[i]] for i in indices.flatten()]
print(f"text: {text}")

text: ['dog', 'person']


## Show image output

In [28]:
font = cv2.FONT_HERSHEY_SIMPLEX
text_color = (255, 255, 255)

# Draw the bounding boxes and labels on the image
for i in range(len(boxes)):
    if i in indices:
        box = boxes[i]
        x, y, w, h = box
        label = str(classes[class_ids[i]])
        conf = confidences[i]               # confidence score
        label = f"{label} {round(conf*100,2)}"       # format the label with confidence score
        color = colors[class_ids[i]]         # Get the color for the class
        
        t_size = cv2.getTextSize(label, font, 1, 1)[0]
        # Draw the bounding box
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        

        # Put the label and confidence on the image
        c2 = x + int(t_size[0]*0.8), y - int(t_size[1]*1.2)             # calculate the position for the label
        cv2.rectangle(img, (x, y-2), c2, color, -1)                     # background rectangle for text
        cv2.putText(img, label, (x, y - 2), font, 0.7, text_color, 3)   # put the label on the image

# Show the image with bounding boxes
cv2.imshow("Image", img)
cv2.waitKey(0)  # Wait for a key press to close the window
cv2.destroyAllWindows()  # Close all OpenCV windows